# 00 — Comparación de modelos y análisis de negocio

Este notebook reúne resultados ya obtenidos en los notebooks de entrenamiento y construye el análisis de valor del sistema. No entrena modelos ni depende de la ejecución de los demás notebooks: las métricas y matrices se registran manualmente después de validar cada experimento.

In [31]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

## Resultados técnicos

La tabla se completa con valores fijos para que este notebook funcione como síntesis independiente. `cv_roc_auc` corresponde a validación cruzada agrupada sobre 2015–2016 y `validation_roc_auc` al holdout temporal de enero–agosto de 2017.

In [32]:
model_results = pd.DataFrame([
    {
        "experiment": "00_dummy_prior",
        "model": "DummyClassifier",
        "feature_set": "No predictive features",
        "cv_roc_auc": np.nan,
        "cv_std": np.nan,
        "validation_roc_auc": 0.5000,
        "accuracy": 0.6130,
        "precision": 0.0000,
        "recall": 0.0000,
        "f1": 0.0000,
        "threshold": 0.5,
    },
    {
        "experiment": "01_logistic_base",
        "model": "Logistic Regression",
        "feature_set": "Original cleaned",
        "cv_roc_auc": 0.8925,
        "cv_std": 0.0036,
        "validation_roc_auc": 0.8422,
        "accuracy": 0.7061,
        "precision": 0.5798,
        "recall": 0.8740,
        "f1": 0.6971,
        "threshold": 0.5,
    },
    {
        "experiment": "02_logistic_engineered",
        "model": "Logistic Regression",
        "feature_set": "Engineered linear set",
        "cv_roc_auc": 0.908557,
        "cv_std": 0.005873,
        "validation_roc_auc": 0.855017,
        "accuracy": 0.739032,
        "precision": 0.620290,
        "recall": 0.839568,
        "f1": 0.713461,
        "threshold": 0.5,
    },
    {
        "experiment": "03_decision_tree_pruned",
        "model": "Decision Tree",
        "feature_set": "Tree-pruned | 17 features",
        "cv_roc_auc": 0.927092,
        "cv_std": 0.002390,
        "validation_roc_auc": 0.838694,
        "accuracy": 0.765822,
        "precision": 0.723456,
        "recall": 0.639187,
        "f1": 0.678716,
        "threshold": 0.5,
    },
    {
        "experiment": "04_random_forest",
        "model": "Random Forest",
        "feature_set": "Cleaned base | 24 features",
        "cv_roc_auc": 0.941260,
        "cv_std": 0.003604,
        "validation_roc_auc": 0.842641,
        "accuracy": 0.769312,
        "precision": 0.748885,
        "recall": 0.607621,
        "f1": 0.670898,
        "threshold": 0.5,
    },
    {
        "experiment": "05_xgboost_regularized",
        "model": "XGBoost",
        "feature_set": "Tree-pruned | 17 features",
        "cv_roc_auc": 0.938277,
        "cv_std": 0.002909,
        "validation_roc_auc": 0.867291,
        "accuracy": 0.781724,
        "precision": 0.737476,
        "recall": 0.676913,
        "f1": 0.705898,
        "threshold": 0.5,
    },
    {
        "experiment": "06_knn_sample_search",
        "model": "KNN",
        "feature_set": "Tree-pruned | 17 features; sample CV",
        "cv_roc_auc": 0.876338,
        "cv_std": 0.008868,
        "validation_roc_auc": 0.825042,
        "accuracy": 0.759751,
        "precision": 0.743475,
        "recall": 0.578914,
        "f1": 0.650955,
        "threshold": 0.5,
    },
    {
        "experiment": "07_mlp_final",
        "model": "MLPClassifier",
        "feature_set": "Tree-pruned | 17 features",
        "cv_roc_auc": 0.936075,
        "cv_std": 0.002251,
        "validation_roc_auc": 0.845721,
        "accuracy": 0.627129,
        "precision": 0.509632,
        "recall": 0.964433,
        "f1": 0.666872,
        "threshold": 0.5,
    },
])
display(model_results)

,experiment,model,feature_set,cv_roc_auc,cv_std,validation_roc_auc,accuracy,precision,recall,f1,threshold
0,00_dummy_prior,DummyClassifier,No predictive features,NaN,NaN,0.500000,0.613000,0.000000,0.000000,0.000000,0.5
1,01_logistic_base,Logistic Regression,Original cleaned,0.892500,0.003600,0.842200,0.706100,0.579800,0.874000,0.697100,0.5
2,02_logistic_engineered,Logistic Regression,Engineered linear set,0.908557,0.005873,0.855017,0.739032,0.620290,0.839568,0.713461,0.5
3,03_decision_tree_pruned,Decision Tree,Tree-pruned | 17 features,0.927092,0.002390,0.838694,0.765822,0.723456,0.639187,0.678716,0.5
4,04_random_forest,Random Forest,Cleaned base | 24 features,0.941260,0.003604,0.842641,0.769312,0.748885,0.607621,0.670898,0.5
5,05_xgboost_regularized,XGBoost,Tree-pruned | 17 features,0.938277,0.002909,0.867291,0.781724,0.737476,0.676913,0.705898,0.5
6,06_knn_sample_search,KNN,Tree-pruned | 17 features; sample CV,0.876338,0.008868,0.825042,0.759751,0.743475,0.578914,0.650955,0.5
7,07_mlp_final,MLPClassifier,Tree-pruned | 17 features,0.936075,0.002251,0.845721,0.627129,0.509632,0.964433,0.666872,0.5


## Hallazgos operativos a igual capacidad

Los modelos fueron entrenados para predecir cancelación general. Para evaluar su utilidad operativa se ordenaron las reservas de 2017 por score y se compararon capacidades idénticas. `critical_recall` mide la cobertura de No-Show y cancelaciones entre cero y siete días; `gross_value_capture` mide la proporción de su valor bruto, no una pérdida contable.


In [33]:
operational_capacity_results = pd.DataFrame([
    {"model": "Logistic Regression", "capacity": 0.05, "critical_recall": 0.0426, "gross_value_capture": 0.0816, "lift_vs_random": 0.8508},
    {"model": "Logistic Regression", "capacity": 0.10, "critical_recall": 0.0720, "gross_value_capture": 0.1143, "lift_vs_random": 0.7196},
    {"model": "Logistic Regression", "capacity": 0.20, "critical_recall": 0.2078, "gross_value_capture": 0.2600, "lift_vs_random": 1.0387},
    {"model": "Logistic Regression", "capacity": 0.30, "critical_recall": 0.3254, "gross_value_capture": 0.3869, "lift_vs_random": 1.0846},
    {"model": "XGBoost", "capacity": 0.05, "critical_recall": 0.0025, "gross_value_capture": 0.0168, "lift_vs_random": 0.0500},
    {"model": "XGBoost", "capacity": 0.10, "critical_recall": 0.0357, "gross_value_capture": 0.0647, "lift_vs_random": 0.3567},
    {"model": "XGBoost", "capacity": 0.20, "critical_recall": 0.1189, "gross_value_capture": 0.1919, "lift_vs_random": 0.5944},
    {"model": "XGBoost", "capacity": 0.30, "critical_recall": 0.2760, "gross_value_capture": 0.3398, "lift_vs_random": 0.9198},
    {"model": "MLP", "capacity": 0.05, "critical_recall": 0.0094, "gross_value_capture": 0.0498, "lift_vs_random": 0.1877},
    {"model": "MLP", "capacity": 0.10, "critical_recall": 0.0350, "gross_value_capture": 0.0938, "lift_vs_random": 0.3504},
    {"model": "MLP", "capacity": 0.20, "critical_recall": 0.1327, "gross_value_capture": 0.2263, "lift_vs_random": 0.6633},
    {"model": "MLP", "capacity": 0.30, "critical_recall": 0.2146, "gross_value_capture": 0.3363, "lift_vs_random": 0.7154},
])
display(operational_capacity_results)


,model,capacity,critical_recall,gross_value_capture,lift_vs_random
0,Logistic Regression,0.05,0.0426,0.0816,0.8508
1,Logistic Regression,0.10,0.0720,0.1143,0.7196
2,Logistic Regression,0.20,0.2078,0.2600,1.0387
3,Logistic Regression,0.30,0.3254,0.3869,1.0846
4,XGBoost,0.05,0.0025,0.0168,0.0500
5,XGBoost,0.10,0.0357,0.0647,0.3567
6,XGBoost,0.20,0.1189,0.1919,0.5944
7,XGBoost,0.30,0.2760,0.3398,0.9198
8,MLP,0.05,0.0094,0.0498,0.1877
9,MLP,0.10,0.0350,0.0938,0.3504


## Diagnóstico del objetivo operativo

ROC AUC para `Cancelled` confirma el valor técnico de los modelos. Al reemplazar retrospectivamente el target por evento crítico, el rendimiento cae de forma marcada. La tercera columna compara solamente cancelaciones críticas contra cancelaciones no críticas.


In [34]:
operational_auc_results = pd.DataFrame([
    {"model": "Logistic Regression", "general_cancellation_roc_auc": 0.855017, "critical_vs_all_roc_auc": 0.565019, "critical_within_cancellations_roc_auc": 0.293529},
    {"model": "XGBoost", "general_cancellation_roc_auc": 0.867291, "critical_vs_all_roc_auc": 0.563477, "critical_within_cancellations_roc_auc": 0.251317},
    {"model": "MLP", "general_cancellation_roc_auc": 0.845721, "critical_vs_all_roc_auc": 0.497290, "critical_within_cancellations_roc_auc": 0.219339},
])
display(operational_auc_results)


,model,general_cancellation_roc_auc,critical_vs_all_roc_auc,critical_within_cancellations_roc_auc
0,Logistic Regression,0.855017,0.565019,0.293529
1,XGBoost,0.867291,0.563477,0.251317
2,MLP,0.845721,0.497290,0.219339


## Política preventiva propuesta

La política considerada consiste en contactar las reservas seleccionadas para solicitar confirmación y, cuando corresponda, una garantía de pago o prepago. La acción no necesariamente evita una cancelación: puede asegurar parte del valor o hacer que el cliente cancele antes, liberando inventario con mayor anticipación.

El contacto también puede producir un efecto adverso: generar fricción y provocar la cancelación de una reserva que de otro modo se habría concretado. El dataset no permite estimar esa cancelación inducida. Por ahora se documenta como limitación y posible costo de falso positivo, sin incorporarla a los cálculos.


## Antecedente: evaluación del baseline

El análisis inicial dependía del umbral y de la matriz de confusión. Los conteos siguientes corresponden al baseline sobre el holdout temporal con umbral 0,5 y se conservan como antecedente. La evaluación final reemplazó esta comparación por políticas de igual capacidad aplicadas a probabilidades individuales.

In [35]:
classification_outcomes = pd.DataFrame([
    {
        "experiment": "01_logistic_base",
        "threshold": 0.5,
        "true_negatives": 14967,
        "false_positives": 9975,
        "false_negatives": 1984,
        "true_positives": 13761,
    },
])
display(classification_outcomes)

,experiment,threshold,true_negatives,false_positives,false_negatives,true_positives
0,01_logistic_base,0.5,14967,9975,1984,13761


## Resultados operativos de cancelaciones críticas

Se considera crítico un `No-Show` o una cancelación registrada entre cero y siete días antes de la llegada. El modelo continúa prediciendo cancelaciones generales; esta clasificación se utiliza únicamente para evaluar qué parte de los eventos más perjudiciales logra identificar.

El valor bruto estimado de una reserva se calcula como `ADR × TotalNights`. Se excluyen de la valoración el registro con `ADR = 5400` y el ADR negativo. Esta medida no equivale a una pérdida contable observada. Los valores se copian del bloque de evaluación operativa del notebook baseline después de ejecutarlo.

In [36]:
critical_cancellation_results = pd.DataFrame([
    {
        "experiment": "01_logistic_base",
        "late_cancellation_days": 7,
        "critical_events": 1598,
        "critical_events_detected": 1179,
        "critical_events_missed": 419,
        "interventions": 23736,
        "non_critical_interventions": 22557,
        "estimated_critical_value": 540098.62,
        "estimated_critical_value_detected": 419003.64,
        "estimated_critical_value_missed": 121094.98,
    },
])
critical_cancellation_results["critical_recall"] = (
    critical_cancellation_results["critical_events_detected"]
    / critical_cancellation_results["critical_events"]
)
critical_cancellation_results["critical_intervention_precision"] = (
    critical_cancellation_results["critical_events_detected"]
    / critical_cancellation_results["interventions"]
)
critical_cancellation_results["critical_value_capture"] = (
    critical_cancellation_results["estimated_critical_value_detected"]
    / critical_cancellation_results["estimated_critical_value"]
)
display(critical_cancellation_results.round(4))

,experiment,late_cancellation_days,critical_events,critical_events_detected,critical_events_missed,interventions,non_critical_interventions,estimated_critical_value,estimated_critical_value_detected,estimated_critical_value_missed,critical_recall,critical_intervention_precision,critical_value_capture
0,01_logistic_base,7,1598,1179,419,23736,22557,540098.62,419003.64,121094.98,0.7378,0.0497,0.7758


## Comparación con intervención aleatoria

Una referencia más informativa que no hacer nada consiste en seleccionar aleatoriamente la misma cantidad de reservas que marca el modelo. Bajo selección aleatoria, la proporción esperada de eventos críticos detectados coincide con la proporción intervenida del holdout. El lift compara la cobertura real del modelo con esa expectativa.

In [37]:
validation_rows = int(classification_outcomes[[
    "true_negatives", "false_positives", "false_negatives", "true_positives"
]].sum(axis=1).iloc[0])
critical_row = critical_cancellation_results.iloc[0]
intervention_rate = critical_row["interventions"] / validation_rows
random_expected_critical_events = critical_row["critical_events"] * intervention_rate
critical_capture_lift = (
    critical_row["critical_events_detected"] / random_expected_critical_events
)

operational_comparison = pd.DataFrame([
    {
        "strategy": "Random selection",
        "interventions": critical_row["interventions"],
        "intervention_rate": intervention_rate,
        "critical_events_detected": random_expected_critical_events,
        "critical_event_recall": intervention_rate,
        "lift_vs_random": 1.0,
    },
    {
        "strategy": "Logistic Regression (threshold 0.5)",
        "interventions": critical_row["interventions"],
        "intervention_rate": intervention_rate,
        "critical_events_detected": critical_row["critical_events_detected"],
        "critical_event_recall": critical_row["critical_recall"],
        "lift_vs_random": critical_capture_lift,
    },
])
display(operational_comparison.round(4))

,strategy,interventions,intervention_rate,critical_events_detected,critical_event_recall,lift_vs_random
0,Random selection,23736,0.5834,932.2419,0.5834,1.0000
1,Logistic Regression (threshold 0.5),23736,0.5834,1179.0000,0.7378,1.2647


## Modelo de valor esperado inicial

Este bloque documenta la primera formulación utilizada para comprobar la mecánica del cálculo sobre el baseline. Se conserva por trazabilidad, pero sus supuestos extremos fueron reemplazados por el análisis individual y los tres escenarios de `11_operational_analysis.ipynb`. El modelo predictivo no evita cancelaciones por sí mismo. Las funciones separan explícitamente:

- la pérdida media producida por una cancelación;
- el costo de aplicar la intervención;
- un posible costo adicional por intervenir innecesariamente sobre un falso positivo;
- y la proporción de cancelaciones detectadas que la acción consigue evitar.

También se incorpora una fracción no recuperada del valor bruto. Una fracción igual a uno representa el supuesto extremo de que una cancelación crítica pierde todo el valor y la habitación no puede revenderse. Los parámetros no deben presentarse como hechos observados: son supuestos para análisis de sensibilidad.

In [38]:
def validate_business_parameters(parameters):
    required = {
        "unrecovered_value_fraction",
        "cost_per_intervention",
        "false_positive_extra_cost",
        "intervention_effectiveness",
    }
    missing = sorted(required.difference(parameters))
    if missing:
        raise ValueError(f"Faltan parámetros de negocio: {missing}")
    if any(parameters[key] < 0 for key in required):
        raise ValueError("Los costos, fracciones y efectividad no pueden ser negativos.")
    for key in ["unrecovered_value_fraction", "intervention_effectiveness"]:
        if parameters[key] > 1:
            raise ValueError(f"{key} debe estar entre 0 y 1.")


def calculate_model_policy(critical_results, parameters):
    validate_business_parameters(parameters)
    interventions = float(critical_results["interventions"])
    non_critical_interventions = float(critical_results["non_critical_interventions"])
    total_critical_value = float(critical_results["estimated_critical_value"])
    detected_critical_value = float(critical_results["estimated_critical_value_detected"])
    missed_critical_value = float(critical_results["estimated_critical_value_missed"])

    unrecovered_fraction = parameters["unrecovered_value_fraction"]
    effectiveness = parameters["intervention_effectiveness"]
    no_action_cost = total_critical_value * unrecovered_fraction
    intervention_cost = interventions * parameters["cost_per_intervention"]
    false_positive_cost = (
        non_critical_interventions * parameters["false_positive_extra_cost"]
    )
    remaining_cancellation_cost = unrecovered_fraction * (
        missed_critical_value + detected_critical_value * (1 - effectiveness)
    )
    model_policy_cost = intervention_cost + false_positive_cost + remaining_cancellation_cost
    expected_savings = no_action_cost - model_policy_cost

    return {
        "no_action_cost": no_action_cost,
        "model_policy_cost": model_policy_cost,
        "expected_savings": expected_savings,
        "relative_savings": expected_savings / no_action_cost if no_action_cost else np.nan,
        "interventions": interventions,
        "expected_value_recovered": detected_critical_value * unrecovered_fraction * effectiveness,
        "interventions": interventions,
    }


def calculate_break_even_effectiveness(critical_results, parameters):
    validate_business_parameters(parameters)
    interventions = float(critical_results["interventions"])
    non_critical_interventions = float(critical_results["non_critical_interventions"])
    detected_value = float(critical_results["estimated_critical_value_detected"])
    avoidable_loss = detected_value * parameters["unrecovered_value_fraction"]
    if avoidable_loss == 0:
        return np.inf
    policy_cost_before_effect = (
        interventions * parameters["cost_per_intervention"]
        + non_critical_interventions * parameters["false_positive_extra_cost"]
    )
    return policy_cost_before_effect / avoidable_loss

## Escenario histórico ilustrativo

Los parámetros siguientes representan la primera prueba, en la que no se recupera ninguna parte del valor de las cancelaciones críticas. No constituyen el resultado económico final y se mantienen exclusivamente como antecedente metodológico.

In [39]:
business_parameters = {
    "unrecovered_value_fraction": 1.0,
    "cost_per_intervention": 10.0,
    "false_positive_extra_cost": 0.0,
    "intervention_effectiveness": 0.25,
}

selected_critical_result = critical_cancellation_results.loc[
    critical_cancellation_results["experiment"].eq("01_logistic_base")
].iloc[0]

required_result_columns = [
    "non_critical_interventions", "estimated_critical_value",
    "estimated_critical_value_detected", "estimated_critical_value_missed",
]
if selected_critical_result[required_result_columns].isna().any():
    print("Análisis pendiente: ejecutar el bloque operativo del baseline y copiar sus resultados.")
else:
    business_result = calculate_model_policy(selected_critical_result, business_parameters)
    business_result["break_even_effectiveness"] = calculate_break_even_effectiveness(
        selected_critical_result, business_parameters
    )
    display(pd.DataFrame([business_result]).round(4))

,no_action_cost,model_policy_cost,expected_savings,relative_savings,interventions,expected_value_recovered,break_even_effectiveness
0,540098.62,672707.71,-132609.09,-0.2455,23736.0,104750.91,0.5665


## Transición desde el baseline hacia la evaluación final

El escenario basado en una única matriz permitió verificar la lógica. El análisis final se realizó luego con probabilidades individuales para:

- comparar diferentes umbrales;
- limitar la cantidad de intervenciones según la capacidad del hotel;
- comparar la priorización del modelo con una selección aleatoria;
- evaluar varios costos y niveles de efectividad;
- y representar la incertidumbre de los supuestos mediante tres escenarios.

La política sólo se considera potencialmente conveniente cuando produce ahorro esperado positivo y mantiene una cantidad de intervenciones operativamente viable.

## Conclusión provisional del baseline

La regresión logística base contiene señal útil: supera la selección aleatoria con igual cantidad de intervenciones y captura una proporción importante de los eventos críticos y de su valor bruto estimado. Sin embargo, con umbral 0,5 requiere intervenir sobre el 58,34 % del holdout y sólo el 4,97 % de esas intervenciones corresponde a un evento crítico. Bajo los parámetros económicos ilustrativos actuales, esta política no resulta conveniente.

Esta conclusión se limita al modelo baseline, al umbral y a los supuestos analizados. No demuestra que todo uso de modelos predictivos carezca de valor; por ese motivo, las etapas posteriores evaluaron Feature Engineering, otros algoritmos y políticas de capacidad limitada.

## Decisiones posteriores al baseline

El análisis de redundancia entre predictores no mostró multicolinealidad preocupante dentro del bloque numérico: el mayor VIF fue 1,52. Sin embargo, `ArrivalDateMonth` y `ArrivalDateWeekNumber` presentan una asociación casi determinística ($\eta=0{,}9951$). Para los conjuntos mejorados se elimina `ArrivalDateWeekNumber` y se conservan mes, año y día. El baseline histórico mantiene sus 25 features originales para preservar la trazabilidad de sus resultados.

Para Logistic Regression tampoco se incorporan `TotalGuests` ni `TotalNights`: no mejoraron ROC AUC y producen multicolinealidad perfecta con sus componentes.

La ablación seleccionó un primer lote parsimonioso de 25 features. `PreviousCancellations` se reemplaza por `PreviousCancellationRate`, y `DaysInWaitingList` por `WasOnWaitingList` y `CityHotelWasOnWaitingList`. No se incorporan `IsDomestic` ni `HasPreviousCancellation`. El conjunto alcanza ROC AUC CV 0,893810 y supera las referencias en los cinco folds; el candidato redundante llega a 0,894158, una diferencia de sólo 0,000348 con cuatro variables adicionales y mayor multicolinealidad. El holdout 2017 no se utilizó para esta selección.

## Selección de features para Logistic Regression

El segundo lote confirmó una relación no lineal importante entre `LeadTime` y la cancelación. La configuración de mayor media conservó `LeadTime`, agregó `LogLeadTime` y `HasSpecialRequests`, alcanzando ROC AUC CV 0,908554. La alternativa que reemplaza `LeadTime` por siete bandas operativas y agrega `HasSpecialRequests` obtuvo 0,908447, apenas 0,000107 menos, con menor dispersión.

Se selecciona la alternativa por bandas para la versión seria de Logistic Regression: permite explicar el riesgo mediante intervalos de anticipación, evita conservar simultáneamente dos representaciones muy asociadas y mejoró la referencia parsimoniosa en los cinco folds. Esta decisión es específica del modelo lineal.

## Resultado de Logistic Regression mejorada

La búsqueda seleccionó `C=1`, pesos balanceados y regularización L1. El modelo alcanzó ROC AUC CV 0,908557 y ROC AUC 0,855017 sobre el holdout temporal, una mejora de 0,012817 frente al baseline. También mejoraron Accuracy, Precision y F1; Recall descendió de 0,874000 a 0,839568 con threshold 0,5.

La caída entre CV y 2017 es 0,053540, por lo que persiste un gap de generalización temporal aunque la mejora se conserva fuera de muestra. Las bandas de `LeadTime` presentaron un patrón monotónico y estable: el riesgo aumenta con la anticipación de la reserva. L1 anuló 79 coeficientes transformados. Los coeficientes extremos de categorías con muy pocos registros no se interpretan como evidencia robusta de importancia. La versión se incorporó posteriormente a la comparación operativa a igual capacidad.

## Resultado del árbol de decisión

La importancia por impureza y la permutación entre folds identificaron como núcleo a `Country`, `MarketSegment`, `PreviousCancellations`, `LeadTime`, `TotalOfSpecialRequests`, `ArrivalDateYear`, `RequiredCarParkingSpaces` y `CustomerType`. La ablación seleccionó una poda de 17 features que mejoró los cinco folds frente a la referencia de 24.

La búsqueda final eligió profundidad 14 y `min_samples_leaf=75`, con ROC AUC CV 0,927092. Sin embargo, en 2017 descendió a 0,838694 y produjo un gap de 0,088398. El árbol alcanzó mayor Accuracy y Precision, pero perdió Recall y F1 frente a Logistic Regression mejorada. Se conserva como herramienta de análisis de variables, no como mejor modelo temporal. No se reajustará utilizando el resultado del holdout.

## Resultado de Random Forest

La base depurada de 24 features superó la poda del árbol en cuatro de cinco folds (0,926422 frente a 0,925012), mostrando que variables poco útiles para un árbol individual pueden aportar diversidad al ensemble. La búsqueda amplia localizó como mejor región profundidad libre, `max_features=0.5` y `min_samples_leaf=5`. Una grilla de confirmación con 100 árboles reprodujo la elección y alcanzó ROC AUC CV 0,941260.

Sobre 2017 el ROC AUC descendió a 0,842641, con un gap de 0,098618. Random Forest logró Accuracy 0,769312 y Precision 0,748885, pero Recall 0,607621 y F1 0,670898. La reducción de 300 a 100 árboles modificó menos de 0,001 el ROC AUC temporal, por lo que la búsqueda económica fue suficiente. Logistic Regression engineered continúa siendo el mejor modelo temporal.

## Resultado de XGBoost

La búsqueda se cerró antes de abrir el holdout y seleccionó una configuración regularizada con `max_depth=7` y `min_child_weight=15`. Alcanzó ROC AUC CV 0,938277 y ROC AUC 0,867291 sobre 2017, el mejor resultado temporal hasta esta etapa. También obtuvo Accuracy 0,781724, Precision 0,737476, Recall 0,676913 y F1 0,705898.

El gap CV–2017 fue 0,070986, menor que en Decision Tree y Random Forest. XGBoost ofrece el mejor ordenamiento general del riesgo, mientras que Logistic Regression mantiene el mayor Recall y un F1 ligeramente superior con threshold 0,5. La comparación operativa definitiva requiere ajustar el threshold y aplicar el mismo análisis económico a las probabilidades individuales de ambos modelos.


## Resultado de KNN

KNN se evaluó como representante de los modelos basados en distancia. Para controlar su costo, la búsqueda se realizó con tres folds sobre una muestra agrupada de 27.137 reservas de desarrollo; por eso su ROC AUC CV de 0,876338 no es estrictamente comparable con las CV completas de los demás modelos. Se seleccionaron 75 vecinos ponderados por distancia.

Sobre 2017 alcanzó ROC AUC 0,825042, Accuracy 0,759751, Precision 0,743475, Recall 0,578914 y F1 0,650955. Aunque conserva una Precision alta, presenta el menor ROC AUC temporal y omite más de cuatro de cada diez cancelaciones. El resultado es consistente con la dificultad de medir cercanía en el espacio de alta dimensión generado por las variables categóricas. No se amplía su búsqueda y se conserva como comparación metodológica, no como candidato final.


## Resultado de la red neuronal MLP

La MLP se optimizó paso a paso sobre 2015-2016. Se seleccionaron una capa de 32 neuronas, early stopping, `alpha=0.01`, learning rate 0,001 y batch 64. La configuración alcanzó ROC AUC CV 0,936075. La validación interna usada por early stopping es aleatoria, no respeta grupos y decide mediante Accuracy; la selección externa se mantuvo agrupada y basada en ROC AUC.

En 2017 la red obtuvo ROC AUC 0,845721 y un gap de 0,090354. Con threshold 0,5 alcanzó Recall 0,964433, pero Precision 0,509632: detectó 15.185 cancelaciones y produjo 14.611 falsos positivos. Su alto Recall puede resultar interesante bajo costos muy asimétricos, aunque su capacidad de ordenamiento es inferior a XGBoost y Logistic Regression. No se reabre la búsqueda con información del holdout.

## Cierre de la etapa de modelos

XGBoost queda como candidato técnico principal por su ROC AUC temporal de 0,867291 y su equilibrio general. Logistic Regression mejorada conserva el mayor F1 y un Recall alto con una estructura interpretable. La MLP maximiza Recall con threshold 0,5, pero requiere demasiadas intervenciones. La evaluación posterior comparó Logistic Regression, XGBoost y MLP mediante políticas de igual capacidad, sin volver a seleccionar hiperparámetros sobre 2017.


## Síntesis del análisis económico final

Los resultados siguientes resumen las alternativas más informativas del análisis individual realizado en `11_operational_analysis.ipynb`. Los importes son ahorros netos esperados bajo supuestos, no ahorros observados.


In [40]:
final_economic_summary = pd.DataFrame([
    {"scenario": "Conservative", "model": "Logistic Regression", "capacity": 0.05, "interventions": 2035, "net_savings": -19583.77, "break_even_loss_reduction": 2.6559},
    {"scenario": "Intermediate", "model": "Logistic Regression", "capacity": 0.05, "interventions": 2035, "net_savings": -4744.17, "break_even_loss_reduction": 0.5621},
    {"scenario": "Favorable extreme", "model": "Logistic Regression", "capacity": 0.05, "interventions": 2035, "net_savings": 10991.97, "break_even_loss_reduction": 0.1351},
    {"scenario": "Favorable extreme", "model": "Logistic Regression", "capacity": 0.30, "interventions": 12207, "net_savings": 48443.20, "break_even_loss_reduction": 0.1675},
])
display(final_economic_summary)


,scenario,model,capacity,interventions,net_savings,break_even_loss_reduction
0,Conservative,Logistic Regression,0.05,2035,-19583.77,2.6559
1,Intermediate,Logistic Regression,0.05,2035,-4744.17,0.5621
2,Favorable extreme,Logistic Regression,0.05,2035,10991.97,0.1351
3,Favorable extreme,Logistic Regression,0.30,12207,48443.20,0.1675


## Conclusión económica final

El análisis por escenarios reemplaza la interpretación extrema del valor bruto como pérdida total. Se aplicaron márgenes de contribución, fracciones no recuperadas diferentes para No-Show y cancelaciones tardías, efectividad de reducción de pérdida y costos de contacto.

Ninguno de los modelos produjo ahorro en los escenarios conservador e intermedio. La alternativa más cercana al equilibrio fue Logistic Regression con capacidad de 5 % en el escenario intermedio: -4.744,17 unidades de ahorro neto y una reducción de pérdida de equilibrio de 56,21 %, frente al 30 % supuesto.

Los resultados fueron positivos principalmente bajo el escenario favorable extremo. Logistic Regression obtuvo el mejor resultado económico, aunque XGBoost mantuvo el mejor ROC AUC para cancelación general. Esto confirma que el mejor modelo técnico para `Cancelled` no es necesariamente el que mejor prioriza los eventos críticos.

El posible costo de descuentos innecesarios, fricción o cancelaciones inducidas por el contacto no pudo estimarse y se mantuvo en cero; incorporarlo sólo reduciría los ahorros calculados. Por tanto, la política no se presenta como una fuente de ahorro demostrada, sino como una posibilidad condicionada a supuestos favorables. Un modelo entrenado específicamente para `IsCriticalCancellation` constituye la principal extensión futura.
